In [ ]:
"""Signal quality overview figure (SCI per channel, SNR and CV), ID vs. SD.

Input folder must contain all_runs_coupling_metrics.csv and fNIRSQA_output.csv
(outputs of compute_snr_cv.py). Output: PDF and PNG.
"""
import tkinter as tk
from pathlib import Path
from tkinter import filedialog

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib import rcParams
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from PIL import Image

# Figure settings following the Neurophotonics guidelines
# https://www.spiedigitallibrary.org/journals/neurophotonics/author-guidelines
rcParams['font.size'] = 8
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
rcParams['figure.facecolor'] = 'white'
rcParams['axes.facecolor'] = 'white'
rcParams['savefig.facecolor'] = 'white'

COLORS = {
    "ID_runA": "#3C7069", "ID_runB": "#9CC4BE",
    "SD_runA": "#5B6B84", "SD_runB": "#B4BED0",
    "neutral_dark": "#4A4A4A", "neutral": "#9A9A9A",
    "area_ID": "#E3EFEC", "area_SD": "#E4E8EE",
    "threshold": "#D62728",
}
COND_COLOR = {"ID": COLORS["ID_runA"], "SD": COLORS["SD_runA"]}
run_palettes = {c: {"RunA": COLORS[f"{c}_runA"], "RunB": COLORS[f"{c}_runB"]} for c in ("ID", "SD")}

In [ ]:
# Select input and output folders
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)


def ask_folder(title):
    folder = filedialog.askdirectory(title=title, initialdir=Path.cwd())
    if not folder:
        raise ValueError(f"No folder selected ({title}).")
    return Path(folder)


input_folder = ask_folder("Select input folder (coupling metrics and fNIRSQA CSVs)")
output_folder = ask_folder("Select output folder")
root.destroy()
output_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def read_csv(folder, name):
    path = folder / name
    if not path.exists():
        raise FileNotFoundError(f"'{name}' not found in '{folder}'.")
    return pd.read_csv(path)


def add_run_label(df, col):
    """Run 1 -> RunA, run 2 -> RunB, parsed from the file name."""
    df['run_label'] = df[col].str.extract(r'run-(\d+)', expand=False).map({'1': 'RunA', '2': 'RunB'})
    n_missing = df['run_label'].isna().sum()
    if n_missing:
        print(f"Warning: no run label for {n_missing} rows, e.g.:")
        print(df.loc[df['run_label'].isna(), col].head())
    return df


def load_coupling(folder):
    df = read_csv(folder, 'all_runs_coupling_metrics.csv')
    df['file_name'] = df['file_name'].apply(lambda x: x.split('_nirs')[0])
    df = df.rename(columns={'file_name': 'file'})
    df['sub'] = df['file'].str.split('_').str[0]
    df = add_run_label(df, 'file')
    return df[['file', 'sub', 'run_label', 'source_index', 'detector_index',
               'channel_label', 'SCI', 'couplingSNR_dB']]


def load_snr_cv(folder):
    df = read_csv(folder, 'fNIRSQA_output.csv')
    df['sub'] = df['Filename'].str.split('_').str[0]
    df = add_run_label(df, 'Filename')
    df['wavelength'] = df['wavelength'].astype(int)
    return df


df_csnr = load_coupling(input_folder)
df_snr_cv = load_snr_cv(input_folder)

In [ ]:
# One SCI value per participant, channel and run
channel_means = (
    df_csnr.groupby(["sub", "channel_label", "run_label"], as_index=False)["SCI"].mean()
)


def channel_key(label):
    s, d = label.split('-')
    return int(s.lstrip('S')), int(d.lstrip('D'))


id_channels = sorted(df_csnr['channel_label'].unique(), key=channel_key)

# The SD montage (5 sources x 5 detectors, 13 channels) is numbered differently
# from the ID montage (8 x 8, 20 channels). Mapping: SD channel -> physically
# identical ID channel.
SD_TO_ID = {
    "S1-D1": "S1-D1", "S1-D3": "S1-D5", "S1-D5": "S1-D8",
    "S2-D3": "S3-D5", "S2-D4": "S3-D6", "S2-D5": "S3-D8",
    "S3-D1": "S6-D1", "S3-D2": "S6-D3", "S3-D3": "S6-D5",
    "S4-D2": "S7-D3", "S4-D3": "S7-D5", "S4-D4": "S7-D6",
    "S5-D2": "S8-D3",
}
sd_channels = set(SD_TO_ID.values())

not_in_id = sorted(sd_channels - set(id_channels), key=channel_key)
if not_in_id:
    raise ValueError(f"SD channels not found in the data: {not_in_id}")
print(f"ID channels: {len(id_channels)}, SD channels: {len(sd_channels)}")

In [ ]:
# SNR/CV for both conditions (SD = subset of ID channels)
df_snr_cv_all = pd.concat(
    [df_snr_cv.assign(condition="ID"),
     df_snr_cv[df_snr_cv["channel_label"].isin(sd_channels)].assign(condition="SD")],
    ignore_index=True,
)
df_snr_cv_all["group"] = df_snr_cv_all["wavelength"].astype(str) + "_" + df_snr_cv_all["condition"]

wavelengths = sorted(df_snr_cv_all["wavelength"].unique())
violin_order = [f"{wl}_{cond}" for wl in wavelengths for cond in ("ID", "SD")]

In [ ]:
dark = COLORS["neutral_dark"]

sd_order = [ch for ch in id_channels if ch in sd_channels]
idonly_order = [ch for ch in id_channels if ch not in sd_channels]
SPACER = " "  # empty category creates a gap between the two groups
plot_order = sd_order + [SPACER] + idonly_order
n_sd, n_idonly = len(sd_order), len(idonly_order)

# The figure is drawn wider than the final 6.75 in and scaled down in the
# paper. All font sizes and line widths are multiplied by S so that they end
# up at 8 pt / >= 0.5 pt after scaling.
FIG_W, FIG_H = 9.0, 7.5
SCI_HEIGHT = 1.6  # height of the SCI panel relative to the SNR/CV row
S = FIG_W / 6.75
rcParams.update({'font.size': 8*S, 'axes.labelsize': 8*S, 'xtick.labelsize': 8*S,
                 'ytick.labelsize': 8*S, 'legend.fontsize': 8*S, 'axes.linewidth': 1*S,
                 'xtick.major.size': 3.5*S, 'ytick.major.size': 3.5*S,
                 'xtick.major.width': 0.8*S, 'ytick.major.width': 0.8*S})

fig = plt.figure(figsize=(FIG_W, FIG_H))
gs = fig.add_gridspec(2, 2, height_ratios=[SCI_HEIGHT, 1.0], hspace=0.55, wspace=0.28)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

# SCI per channel, grouped into SD and ID-only channels
ax1.axvspan(-0.5, n_sd - 0.5, color=COLORS["area_SD"], zorder=0, lw=0)
ax1.axvspan(n_sd + 0.5, len(plot_order) - 0.5, color=COLORS["area_ID"], zorder=0, lw=0)
ax1.axhline(0.75, ls='--', lw=1.3*S, color=COLORS["threshold"], zorder=3)

# Same category order in both calls keeps the x positions identical
for chans, pal in ((sd_order, run_palettes["SD"]), (idonly_order, run_palettes["ID"])):
    sns.stripplot(
        data=channel_means[channel_means["channel_label"].isin(chans)],
        x="channel_label", y="SCI", order=plot_order,
        size=4*S, alpha=0.85, edgecolor=pal["RunA"], linewidth=0.5*S, dodge=True,
        hue="run_label", hue_order=["RunA", "RunB"], palette=pal,
        ax=ax1, legend=False, zorder=2,
    )

ax1.set_xlabel('')
ax1.set_ylabel('SCI')
ax1.set_yticks([-0.5, 0, 0.5, 0.75, 1.0])
ax1.set_yticklabels(["-0.5", "0", "0.5", "0.75", "1"])
for lab in ax1.get_yticklabels():
    if lab.get_text() == "0.75":
        lab.set_color(COLORS["threshold"])
        lab.set_fontweight('bold')
ax1.set_xlim(-0.5, len(plot_order) - 0.5)
ax1.set_xticks(range(len(plot_order)))
ax1.set_xticklabels(plot_order, rotation=45, ha='right', rotation_mode='anchor')
for tick, ch in zip(ax1.get_xticklabels(), plot_order):
    tick.set_fontweight('bold')
    tick.set_color(COND_COLOR["SD" if ch in sd_channels else "ID"])


def group_bracket(x0, x1, text, color):
    tr = ax1.get_xaxis_transform()  # x in data, y in axes coordinates
    ax1.plot([x0, x0, x1, x1], [1.02, 1.05, 1.05, 1.02], transform=tr,
             color=color, lw=1.5*S, clip_on=False, solid_capstyle='butt')
    ax1.text((x0 + x1) / 2, 1.07, text, transform=tr, ha='center', va='bottom',
             color=color, fontweight='bold')


group_bracket(-0.4, n_sd - 0.6, f"SD channels (n = {n_sd}, subset of ID)", COND_COLOR["SD"])
group_bracket(n_sd + 0.6, len(plot_order) - 0.6, f"ID-only channels (n = {n_idonly})", COND_COLOR["ID"])
ax1.set_title('Mean SCI Values per Participant and Channel', pad=30*S)


def split_violin(metric, ax):
    for cond in ("ID", "SD"):
        sns.violinplot(
            data=df_snr_cv_all[df_snr_cv_all["condition"] == cond],
            x="group", y=metric, order=violin_order,
            hue="run_label", hue_order=["RunA", "RunB"],
            palette=run_palettes[cond],
            linecolor=dark, ax=ax, split=True, inner="quartile", cut=0, legend=False,
            width=0.9, linewidth=0.8*S,
        )
    ax.set_xticks(range(len(violin_order)))
    ax.set_xticklabels([g.split("_")[1] for g in violin_order])
    for tick, g in zip(ax.get_xticklabels(), violin_order):
        tick.set_color(COND_COLOR[g.split("_")[1]])
        tick.set_fontweight('bold')
    for k, wl in enumerate(wavelengths):
        ax.annotate(f"{wl} nm", xy=((2 * k + 1) / len(violin_order), 0), xycoords='axes fraction',
                    xytext=(0, -20*S), textcoords='offset points', ha='center', va='top')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.yaxis.set_major_locator(MultipleLocator(10))
    ax.yaxis.set_minor_locator(MultipleLocator(5))
    ax.tick_params(axis='y', which='minor', length=2*S)


split_violin("SNR", ax2)
ax2.set_title('SNR (dB)')
split_violin("CV", ax3)
ax3.set_title('CV (%)')
ax3.set_ylim(0, df_snr_cv_all["CV"].max() * 1.02)

handles = [
    Patch(facecolor=COLORS["ID_runA"], edgecolor=dark, label="ID, Run A"),
    Patch(facecolor=COLORS["ID_runB"], edgecolor=dark, label="ID, Run B"),
    Patch(facecolor=COLORS["SD_runA"], edgecolor=dark, label="SD, Run A"),
    Patch(facecolor=COLORS["SD_runB"], edgecolor=dark, label="SD, Run B"),
]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.09),
           ncol=4, fontsize=10*S, frameon=True, fancybox=False,
           edgecolor=COLORS["neutral"], borderpad=0.7,
           handlelength=2.2, handleheight=1.3, columnspacing=2.0)

sns.despine()

# PDF (vector, fonts embedded) and PNG (600 dpi, no alpha channel)
fig_path_png = output_folder / 'signal_quality_overview_ID_SD.png'
fig_path_pdf = output_folder / 'signal_quality_overview_ID_SD.pdf'
fig.savefig(fig_path_pdf, bbox_inches='tight', transparent=False)
fig.savefig(fig_path_png, dpi=600, bbox_inches='tight', transparent=False)

with Image.open(fig_path_png) as im:
    if im.mode in ('RGBA', 'LA', 'P'):
        rgba = im.convert('RGBA')
        rgb_im = Image.new('RGB', im.size, (255, 255, 255))
        rgb_im.paste(rgba, mask=rgba.split()[-1])
        rgb_im.save(fig_path_png)

for path in (fig_path_pdf, fig_path_png):
    size_mb = path.stat().st_size / (1024 * 1024)
    flag = "  <-- exceeds 3 MB" if size_mb > 3 else ""
    print(f"{path.name}: {size_mb:.2f} MB{flag}")

plt.show()